In [1]:
# figure_3_1_pipeline.py  —  PhD-presentation refactor
#
# Solar flare forecasting pipeline (schematic).
#
# The pipeline content is unchanged; only the rendering is reworked so the
# figure reads cleanly on a projector / slide:
#   * 16:9 canvas;
#   * six shaded stage columns with header pills (DATA … EVALUATION);
#   * a small Node helper stores each card's geometry, so every connector
#     attaches to a real box edge rather than a hand-tuned coordinate;
#   * uniform card height, a header rule under every title, soft drop shadows;
#   * one typographic hierarchy (title / stage / card-title / body);
#   * an explicit legend separating the data/feature flow from the
#     labels-only link (the flare catalogue never enters the model);
#   * vector output (PDF + SVG) beside the 300-dpi PNG.

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle

plt.rcParams.update({"font.family": "DejaVu Sans", "svg.fonttype": "none"})

# ---------------------------------------------------------------- palette
GOES  = "#1b6ca8"   # soft X-ray data  / GOES processing
SHARP = "#d9822b"   # magnetogram data / SHARP processing
FEAT  = "#2e7d5b"   # feature & label construction
CAT   = "#7048a8"   # flare catalogue  (labels only)
MODEL = "#3c5a73"   # models
EVAL  = "#8c3b32"   # evaluation
PILL  = "#33475b"   # stage header pills
INK   = "#1a1a1a"
MUTE  = "#5a5a5a"
BAND  = ("#f5f6f8", "#edeff2")

# ---------------------------------------------------------------- canvas
fig, ax = plt.subplots(figsize=(16, 9), dpi=300)
ax.set_xlim(0, 190)
ax.set_ylim(0, 96)
ax.axis("off")


# ---------------------------------------------------------------- helpers
class Node:
    """A rounded card that remembers its geometry so connectors can attach
    to its edges instead of hand-tuned coordinates."""

    def __init__(self, cx, cy, w, h):
        self.cx, self.cy, self.w, self.h = cx, cy, w, h

    def left(self, dy=0):   return (self.cx - self.w / 2, self.cy + dy)
    def right(self, dy=0):  return (self.cx + self.w / 2, self.cy + dy)
    def top(self, dx=0):    return (self.cx + dx, self.cy + self.h / 2)
    def bottom(self, dx=0): return (self.cx + dx, self.cy - self.h / 2)


def box(cx, cy, w, h, title, body, fc, fs_t=13, fs_b=9.5):
    patch = FancyBboxPatch((cx - w / 2, cy - h / 2), w, h,
                           boxstyle="round,pad=0,rounding_size=1.4",
                           fc=fc, ec="white", lw=1.2, zorder=3)
    patch.set_path_effects([pe.withSimplePatchShadow(
        offset=(2, -2), shadow_rgbFace="#3a3a3a", alpha=0.18)])
    ax.add_patch(patch)
    top = cy + h / 2
    ax.text(cx, top - 3.6, title, ha="center", va="center", zorder=4,
            fontsize=fs_t, fontweight="bold", color="white")
    ax.plot([cx - w / 2 + 3.5, cx + w / 2 - 3.5], [top - 6.3] * 2,
            color="white", lw=1.0, alpha=0.35, zorder=4)
    ax.text(cx, top - 8.1, body, ha="center", va="top", zorder=4,
            fontsize=fs_b, color="white", linespacing=1.5)
    return Node(cx, cy, w, h)


def flow(p1, p2, color=INK, lw=2.4, dashed=False):
    ax.add_patch(FancyArrowPatch(
        p1, p2, arrowstyle="-|>", mutation_scale=22, lw=lw, color=color,
        linestyle=(0, (5, 3)) if dashed else "-",
        shrinkA=1, shrinkB=2, joinstyle="round", capstyle="round", zorder=5))


# ---------------------------------------------------- stage columns
stages = [
    ("DATA",             3,  32),
    ("PRE-PROCESSING",  32,  65),
    ("FEATURES",        65,  97),
    ("LABELLING",       97, 129),
    ("MODELLING",      129, 159),
    ("EVALUATION",     159, 188),
]
for i, (name, x0, x1) in enumerate(stages):
    ax.add_patch(Rectangle((x0, 18), x1 - x0, 58,
                           fc=BAND[i % 2], ec="none", zorder=0))
    ax.add_patch(FancyBboxPatch((x0 + 2.5, 78.5), (x1 - x0) - 5, 5.0,
                                boxstyle="round,pad=0,rounding_size=2.0",
                                fc=PILL, ec="none", zorder=1))
    ax.text((x0 + x1) / 2, 80.9, name, ha="center", va="center",
            fontsize=11, fontweight="bold", color="white", zorder=2)

# ----------------- cards   (top lane y=61 | bottom lane y=33 | hub y=47)
goes_xrs = box(17, 61, 26, 23, "GOES XRS",
               "0.5–4 Å  &  1–8 Å\nsoft X-ray flux\n1-min cadence", GOES)
sharp    = box(17, 33, 26, 23, "SHARP",
               "vector magnetograms\nactive-region\ndescriptors · 12 min", SHARP)

goes_cln = box(48, 61, 29, 23, "GOES cleaning",
               "despike · interpolate\ncausal smoothing\nbackground-normalise\nresample → 12 min", GOES)
sharp_qc = box(48, 33, 29, 23, "SHARP QC + aggr.",
               "QUALITY = 0, limb ≤ 68°\nsum (extensive) /\nmax (intensive)\nover regions per step", SHARP)

feat = box(81, 47, 31, 23, "Feature build",
           "GOES 6 h window (30×2)\n+ disk-wide SHARP vector\nlags & trailing statistics\n"
           "→  257 features", FEAT, fs_b=9)

cat = box(114, 62, 28, 17, "Flare catalogue",
          "NCEI flare catalogue\npeak time · class ≥ M", CAT, fs_b=9)
lbl = box(114, 35, 28, 23, "Labelled instance",
          "features paired with the\n≥ M / 24 h label\ny(t₀) = 1 if a ≥ M flare in\n(t₀, t₀ + 24 h]",
          FEAT, fs_b=9)

models = box(144, 47, 27, 23, "Models",
             "LSTM (GOES)\nLSTM (GOES + SHARP)\nearly fusion\nclassical baselines", MODEL, fs_b=9)
evalb = box(173.5, 47, 25, 23, "Evaluation",
            "chronological split\n+ 24 h embargo\nTSS · HSS · Brier\nval-only threshold", EVAL, fs_b=9)

# ---------------------------------------------------- connectors
flow(goes_xrs.right(), goes_cln.left(), GOES)
flow(sharp.right(),    sharp_qc.left(), SHARP)
flow(goes_cln.right(), feat.left(dy=+8), GOES)
flow(sharp_qc.right(), feat.left(dy=-8), SHARP)
flow(feat.right(dy=-8), lbl.left(dy=+6), FEAT)
flow(cat.bottom(), lbl.top(), CAT, lw=2.2, dashed=True)          # catalogue -> label ONLY
ax.text(116.5, 49.5, "labels only", ha="left", va="center",
        fontsize=9.5, style="italic", color=CAT)
flow(lbl.right(dy=+12), models.left(), FEAT)
flow(models.right(), evalb.left(), MODEL)

# ---------------------------------------------------- title + legend
ax.text(3, 91, "Solar flare forecasting pipeline", ha="left", va="center",
        fontsize=19, fontweight="bold", color=INK)

flow((6, 8), (13, 8), INK)
ax.text(15, 8, "data / feature flow", ha="left", va="center",
        fontsize=10, color=INK)
flow((66, 8), (73, 8), CAT, lw=2.2, dashed=True)
ax.text(75, 8,
        "labels only  —  the flare catalogue builds the label and never enters the model",
        ha="left", va="center", fontsize=10, color=CAT)
ax.text(95, 3,
        "Figure 3.1    Solar flare forecasting pipeline (schematic).    "
        "One disk-level forecast instance is produced per 12-minute timestamp.",
        ha="center", va="center", fontsize=10, style="italic", color=MUTE)

for ext in ("png", "pdf", "svg"):
    fig.savefig(f"Figure_3_1_pipeline.{ext}",
                dpi=300 if ext == "png" else None,
                bbox_inches="tight", pad_inches=0.3, facecolor="white")


In [2]:
# figure_3_5_disk_aggregation.py
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle

plt.rcParams.update({"font.family": "DejaVu Sans"})

fig, ax = plt.subplots(figsize=(18, 9), dpi=300)
ax.set_xlim(0, 102); ax.set_ylim(0, 62); ax.axis("off")

SHARP = "#e07b2a"; STAGE = "#2f6f4f"; SUM = "#1f6fb4"; MX = "#8e44ad"; GREY = "#555"

def rbox(cx, cy, w, h, fc, ec="#1a1a1a", lw=1.6):
    ax.add_patch(FancyBboxPatch((cx - w/2, cy - h/2), w, h,
                 boxstyle="round,pad=0.4,rounding_size=1.1", fc=fc, ec=ec, lw=lw))

def arrow(x1, y1, x2, y2, color, lw=3.2):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>",
                 mutation_scale=30, lw=lw, color=color, shrinkA=1, shrinkB=1))

# Column headers
ax.text(20, 59.5, "Many HARP records at t₀", ha="center", va="center",
        fontsize=15.5, fontweight="bold", color="#1a1a1a")
ax.text(55, 59.5, "Per-descriptor aggregation", ha="center", va="center",
        fontsize=15.5, fontweight="bold", color="#1a1a1a")
ax.text(88, 59.5, "One disk-wide vector", ha="center", va="center",
        fontsize=15.5, fontweight="bold", color="#1a1a1a")
ax.text(55, 56.4, "(aggregate_sharp_disk)", ha="center", va="center",
        fontsize=12, style="italic", color=GREY)

# LEFT: solar disk with HARP records
ax.add_patch(Circle((20, 31), 15, fc="#fff7ec", ec=SHARP, lw=2.2))
ax.text(20, 47.6, "solar disk", ha="center", va="center",
        fontsize=12.5, style="italic", color=SHARP)
for x, y, name in [(14, 36, "HARP 1"), (27, 34, "HARP 2"),
                   (16, 25, "HARP 3"), (26, 23, "HARP 4")]:
    rbox(x, y, 9, 5.4, SHARP)
    ax.text(x, y + 1.2, name, ha="center", va="center",
            fontsize=11.5, fontweight="bold", color="white")
    ax.text(x, y - 1.7, "18 descr.", ha="center", va="center",
            fontsize=10, color="white")
ax.text(20, 13.6, "QC-accepted, limb ≤ 68° only", ha="center", va="center",
        fontsize=12, color=SHARP)

# MIDDLE: SUM / MAX rule boxes
rbox(55, 43, 27, 10.5, SUM)
ax.text(55, 45.7, "SUM  —  7 extensive totals", ha="center", va="center",
        fontsize=13.5, fontweight="bold", color="white")
ax.text(55, 41.2, "USFLUX, TOTUSJZ, TOTUSJH, ABSNJZH,\nSAVNCPP, TOTPOT, AREA_ACR",
        ha="center", va="center", fontsize=10.8, color="white", linespacing=1.3)

rbox(55, 25, 27, 13.5, MX)
ax.text(55, 29.6, "MAX  —  11 intensive", ha="center", va="center",
        fontsize=13.5, fontweight="bold", color="white")
ax.text(55, 24.2,
        "MEANPOT, R_VALUE, MEANGAM, MEANGBT,\nMEANGBZ, MEANGBH, MEANALP, MEANJZH,\n"
        "MEANJZD, MEANSHR, SHRGT45",
        ha="center", va="center", fontsize=10.6, color="white", linespacing=1.3)
ax.text(55, 14.6, "unweighted, over all surviving regions; no row if none survive",
        ha="center", va="center", fontsize=11.5, style="italic", color=GREY)

# RIGHT: disk-wide 18-vector
rbox(88, 33, 16, 20, STAGE)
ax.text(88, 40, "xₛ(t₀) ∈ ℝ¹⁸", ha="center", va="center",
        fontsize=15.5, fontweight="bold", color="white")
ax.text(88, 32, "7 summed +\n11 max-ed\ndescriptors", ha="center", va="center",
        fontsize=12.5, color="white", linespacing=1.4)
ax.text(88, 25.5, "→ feature build\n(Section 3.2.1)", ha="center", va="center",
        fontsize=11.5, style="italic", color="white", linespacing=1.3)

# Arrows
arrow(35.5, 31, 41, 34, SHARP)
arrow(69, 33, 79.5, 33, STAGE)

ax.text(51, 5.0,
        "Figure 3.5  SHARP disk aggregation (schematic). The SHARP records of all "
        "quality-accepted, on-disk active regions at each",
        ha="center", va="center", fontsize=12.5, color="#1a1a1a")
ax.text(51, 2.2,
        "12-minute timestamp are reduced to a single disk-wide 18-descriptor vector — "
        "summed for extensive quantities, maximum for intensive ones.",
        ha="center", va="center", fontsize=12.5, color="#1a1a1a")

fig.savefig("Figure_3_5_disk_aggregation.png", dpi=300, bbox_inches="tight", facecolor="white")

In [2]:
# figure_3_8_descriptor_scaling.py
# Figure 3.8 — SHARP descriptor preprocessing: raw vs Winsorised+robust-scaled,
# and train/test scaled-distribution overlay. NO synthetic "magnetogram" panel.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 12})

# ----------------------------------------------------------------------
# LOAD YOUR REAL PIPELINE OUTPUT HERE.
# Replace the four arrays below with the actual descriptor values your
# preprocessing produces (e.g. np.load(...) or pandas from the run dir).
#   raw_train   : one raw SHARP descriptor, training split   (1-D array)
#   scaled_train: same descriptor after Winsorise(1,99)+robust scaling, train
#   scaled_test : same descriptor after the *train-fitted* scaler, test split
# Keep it to ONE named descriptor (state which in the caption), e.g. USFLUX.
# ----------------------------------------------------------------------
USE_REAL_DATA = False   # <-- set True once the arrays below are real
DESCRIPTOR = "USFLUX"   # <-- name the descriptor you actually plot

if USE_REAL_DATA:
    raw_train    = np.load("raw_train_USFLUX.npy")
    scaled_train = np.load("scaled_train_USFLUX.npy")
    scaled_test  = np.load("scaled_test_USFLUX.npy")
else:
    # PLACEHOLDER ONLY — for layout preview. Do NOT submit with this.
    rng = np.random.default_rng(0)
    raw_train    = np.concatenate([rng.normal(0, 1, 900), rng.normal(4, 0.6, 80)])
    lo, hi = np.percentile(raw_train, [1, 99])
    w = np.clip(raw_train, lo, hi)
    med, iqr = np.median(w), np.subtract(*np.percentile(w, [75, 25]))
    scaled_train = (w - med) / iqr
    _t = np.concatenate([rng.normal(0.1, 1.05, 700), rng.normal(4, 0.6, 70)])
    _t = np.clip(_t, lo, hi)
    scaled_test = (_t - med) / iqr

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4), dpi=300)

# Panel (a): raw vs Winsorised + robust-scaled (same descriptor)
ax1.hist(raw_train, bins=45, color="#b0b0b0", alpha=0.85, label="Raw")
ax1.hist(scaled_train, bins=45, color="#1f4fb4", alpha=0.75,
         label="Winsorised + robust-scaled")
ax1.set_xlabel(f"{DESCRIPTOR} value")
ax1.set_ylabel("Count")
ax1.set_title("(a) Raw vs Winsorised + robust-scaled (train)", fontsize=12.5)
ax1.legend(frameon=False, fontsize=10.5)
ax1.spines[["top", "right"]].set_visible(False)

# Panel (b): scaled train vs scaled test (train-fitted scaler applied to test)
ax2.hist(scaled_train, bins=45, color="#2e8b57", alpha=0.7, density=True,
         label="Train")
ax2.hist(scaled_test, bins=45, color="#e0a020", alpha=0.7, density=True,
         label="Test")
ax2.set_xlabel(f"scaled {DESCRIPTOR} value")
ax2.set_ylabel("Density")
ax2.set_title("(b) Scaled train vs test (scaler fitted on train only)",
              fontsize=12.5)
ax2.legend(frameon=False, fontsize=10.5)
ax2.spines[["top", "right"]].set_visible(False)

if not USE_REAL_DATA:
    fig.text(0.5, 0.005, "PLACEHOLDER DATA — regenerate from a pipeline run "
             "before submission", ha="center", fontsize=9.5,
             style="italic", color="#b00")

fig.tight_layout()
fig.savefig("Figure_3_8_descriptor_scaling.png", dpi=300,
            bbox_inches="tight", facecolor="white")